In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

N=1000000
square_length = 300 #in cm
sphere_centre = np.array([0,0,-15]) #in cm
sphere_radius = 15 #in cm

x0 = np.zeros(N)
y0 = np.zeros(N)
z0 = np.zeros(N)
theta = np.zeros(N)
phi = np.zeros(N)
dx = np.zeros(N)
dy = np.zeros(N)
dz = np.zeros(N)
b = np.zeros(N)
chord = np.zeros(N)


for muon in range(N):
    x0[muon] = np.random.uniform(-square_length/2, square_length/2)
    y0[muon] = np.random.uniform(-square_length/2, square_length/2)
    z0[muon] = 0


    #angles
    accepted = False
    while not accepted:
        theta_trial = np.random.uniform(-np.pi/2, np.pi/2)
        n = np.random.uniform(0, 1)

        if n < (2/np.pi) * np.cos(theta_trial)**2:
            theta[muon] = theta_trial
            accepted = True

    phi[muon] = np.random.uniform(0,2*np.pi)
    #theta[muon] = 0
    #directions
    dx[muon] = np.sin(theta[muon])*np.cos(phi[muon])
    dy[muon] = np.sin(theta[muon])*np.sin(phi[muon])
    dz[muon] = -np.cos(theta[muon])


    #origins = (x0,y0,z0=0) = O
    #directions = (dx,dy,dz) = D
    #sphere centre = (0,0,-d) 
    O = np.array([x0[muon], y0[muon], z0[muon]])
    D = np.array([dx[muon], dy[muon], dz[muon]])
    O_C = O - sphere_centre

    b[muon] = np.linalg.norm(np.cross(O_C, D))

    chord[muon] = 0

    if b[muon] < sphere_radius:
        chord[muon] = 2*np.sqrt(sphere_radius**2 - b[muon]**2)
    else:
        chord[muon] = 0   



df = pd.DataFrame({
    "x (cm)": x0,
    "y (cm)": y0,
    "z (cm)": z0,
    "theta (rad)": theta,
    "phi (rad)": phi,
    "chord length (cm)": chord,
    "closest approach to sphere centre": b
})

print(df.head())
print(np.sum(chord > 0))

       x (cm)      y (cm)  z (cm)  theta (rad)  phi (rad)  chord length (cm)  \
0   47.677666 -103.088130     0.0    -0.070240   0.858123                0.0   
1   40.537900   90.852613     0.0    -0.180839   2.006053                0.0   
2 -106.971119  128.031704     0.0    -0.019769   2.910635                0.0   
3   99.314374  -97.454004     0.0     0.184735   5.534413                0.0   
4   93.760291  120.413295     0.0     0.564283   6.008375                0.0   

   closest approach to sphere centre  
0                         113.969183  
1                          97.058625  
2                         166.580158  
3                         139.530978  
4                         152.273753  
9984


In [12]:
#two layer on bottom ALIGNED
scintillator1_pos = np.array([0,0,-95])
scintillator2_pos = np.array([2,2,-103])
scint_width = 25
scint_length = 135
scint_thickness = 1


veto_trigger = np.zeros(N, dtype=bool)
for muon in range(N):
    # intersection with scintillator 1
    t1 = (scintillator1_pos[2] - z0[muon]) / dz[muon]
    x1 = x0[muon] + t1*dx[muon]
    y1 = y0[muon] + t1*dy[muon]

    hit1 = False
    if abs(x1) < scint_width/2 and abs(y1) < scint_length/2:
        hit1 = True

    # intersection with scintillator 2
    t2 = (scintillator2_pos[2] - z0[muon]) / dz[muon]
    x2 = x0[muon] + t2*dx[muon]
    y2 = y0[muon] + t2*dy[muon]

    hit2 = False
    if -scint_width/2 + scintillator2_pos[0]<x2<scint_width/2 + scintillator2_pos[0] and -scint_length/2 + scintillator2_pos[1]<y2<scint_length/2+ scintillator2_pos[1]:
        hit2 = True

    # coincidence
    if hit1 and hit2:
        veto_trigger[muon] = True
        


df = pd.DataFrame({
    "x (cm)": x0,
    "y (cm)": y0,
    "z (cm)": z0,
    "theta (rad)": theta,
    "phi (rad)": phi,
    "chord length (cm)": chord,
    "closest approach to sphere centre": b,
    "veto trigger": veto_trigger
})

print(df.head())

print('configuration: both below, aligned')
print('number intersected sphere:', np.sum(chord > 0))
print('number passed through veto:', np.sum(veto_trigger == True))


veto_and_sphere = (chord > 0) & veto_trigger
print('number passed through veto and sphere:', np.sum(veto_and_sphere))

print('number in veto but not sphere:', np.sum(veto_trigger == True)-np.sum(veto_and_sphere))

       x (cm)      y (cm)  z (cm)  theta (rad)  phi (rad)  chord length (cm)  \
0   47.677666 -103.088130     0.0    -0.070240   0.858123                0.0   
1   40.537900   90.852613     0.0    -0.180839   2.006053                0.0   
2 -106.971119  128.031704     0.0    -0.019769   2.910635                0.0   
3   99.314374  -97.454004     0.0     0.184735   5.534413                0.0   
4   93.760291  120.413295     0.0     0.564283   6.008375                0.0   

   closest approach to sphere centre  veto trigger  
0                         113.969183         False  
1                          97.058625         False  
2                         166.580158         False  
3                         139.530978         False  
4                         152.273753         False  
configuration: both below, aligned
number intersected sphere: 9984
number passed through veto: 29773
number passed through veto and sphere: 2414
number in veto but not sphere: 27359


In [13]:
#percentage of muons in sphere which were vetoed

print('Percentage of muons in sphere vetoed:', 100* np.sum(veto_and_sphere)/ np.sum(chord > 0))

Percentage of muons in sphere vetoed: 24.178685897435898
